In [ ]:
#!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
#!pip install --no-cache-dir "trl==0.20.0" xformers vllm gguf pybase64 cbor2

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# Ensures your T4/L4 GPU is active
!nvidia-smi

Mon Mar 30 07:53:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P0             30W /   70W |     155MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.3.17: Fast Qwen2 patching. Transformers: 4.57.6. vLLM: 0.18.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
unsloth/qwen2.5-3b-instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

Unsloth 2026.3.17 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


In [ ]:
SYSTEM_PROMPT = "Count letters. Use <reasoning> and <answer> tags."

In [ ]:
WORDS = ["APPLE", "BANANA", "CHERRY", "DOG", "ELEPHANT"]
data_dict = {"word": WORDS, "length": [len(w) for w in WORDS]}
ds = Dataset.from_dict(data_dict)

In [ ]:
def format_ds(example):
    return {
        "prompt": [{"role": "system", "content": SYSTEM_PROMPT},
                   {"role": "user", "content": f"Word: {example['word']}"}],
        "answer": example['length']
    }
dataset = ds.map(format_ds)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [ ]:
def reward_func(completions, answer, **kwargs):
    # Simple logic: +2.0 for correct digit in answer
    return [2.0 if str(a) in c else 0.0 for c, a in zip(completions, answer)]

In [ ]:
from trl import GRPOConfig
training_args = GRPOConfig(
    learning_rate = 5e-6,
    per_device_train_batch_size = 1,
    num_generations = 4,
    max_steps = 10,
    bf16 = False,
    generation_batch_size = 4,
)

In [ ]:
from trl import GRPOTrainer
trainer = GRPOTrainer(
    model = model,
    reward_funcs = [reward_func],
    args = training_args,
    train_dataset = dataset,
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5 | Num Epochs = 1 | Total steps = 10
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 2 x 1) = 2
 "-____-"     Trainable parameters = 119,734,272 of 3,205,672,960 (3.74% trained)


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_func / mean,rewards / reward_func / std
1,0.000000,0.000000,0.000000,77.250000,52.000000,95.000000,0.000000,77.250000,52.000000,95.000000,0.000063,0.000000,0.000000
2,0.000000,,,,,,,,,,0.000070,,
3,0.000000,0.000000,0.000000,86.500000,54.000000,115.000000,0.000000,86.500000,54.000000,115.000000,0.000059,0.000000,0.000000
4,0.000000,,,,,,,,,,0.000042,,
5,0.000000,0.000000,0.000000,53.750000,43.000000,64.000000,0.000000,53.750000,43.000000,64.000000,0.000038,0.000000,0.000000
6,0.000000,,,,,,,,,,0.000347,,
7,0.000000,0.000000,0.000000,75.000000,53.000000,103.000000,0.000000,75.000000,53.000000,103.000000,0.000235,0.000000,0.000000
8,0.000000,,,,,,,,,,0.000197,,
9,0.000000,0.000000,0.000000,69.750000,40.000000,86.000000,0.000000,69.750000,40.000000,86.000000,0.000065,0.000000,0.000000
10,0.000000,,,,,,,,,,0.000906,,


Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=10, training_loss=1.0117545805599093e-07, metrics={'train_runtime': 211.3945, 'train_samples_per_second': 0.095, 'train_steps_per_second': 0.047, 'total_flos': 0.0, 'train_loss': 1.0117545805599093e-07})

In [ ]:
training_args.max_steps = 100
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5 | Num Epochs = 10 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 2 x 1) = 2
 "-____-"     Trainable parameters = 119,734,272 of 3,205,672,960 (3.74% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / reward_func / mean,rewards / reward_func / std
1,0.000000,0.000000,0.000000,78.750000,50.000000,102.000000,0.000000,78.750000,50.000000,102.000000,0.000950,0.000000,0.000000
2,0.000000,,,,,,,,,,0.000127,,
3,0.000000,0.000000,0.000000,92.250000,77.000000,109.000000,0.000000,92.250000,77.000000,109.000000,0.000122,0.000000,0.000000
4,0.000000,,,,,,,,,,0.000029,,
5,0.000000,0.000000,0.000000,65.500000,62.000000,69.000000,0.000000,65.500000,62.000000,69.000000,0.000036,0.000000,0.000000
6,0.000000,,,,,,,,,,0.000259,,
7,0.000000,0.000000,0.000000,70.750000,40.000000,91.000000,0.000000,70.750000,40.000000,91.000000,0.000319,0.000000,0.000000
8,0.000000,,,,,,,,,,0.000115,,
9,0.000000,0.000000,0.000000,72.500000,44.000000,123.000000,0.000000,72.500000,44.000000,123.000000,0.000044,0.000000,0.000000
10,0.000000,,,,,,,,,,0.000102,,


TrainOutput(global_step=100, training_loss=4.1973788609261933e-07, metrics={'train_runtime': 752.8039, 'train_samples_per_second': 0.266, 'train_steps_per_second': 0.133, 'total_flos': 0.0, 'train_loss': 4.1973788609261933e-07})

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/chat_template.jinja',
 'lora_model/vocab.json',
 'lora_model/merges.txt',
 'lora_model/added_tokens.json',
 'lora_model/tokenizer.json')

In [ ]:
messages = tokenizer.apply_chat_template([
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Word: STRAWBERRY"}
], tokenize = True, add_generation_prompt = True,
return_tensors="pt", return_dict=True)

from vllm import SamplingParams
sampling_params = SamplingParams(temperature = 0.8, top_p = 0.95, max_tokens = 1024)

# Extract parameters from sampling_params object
output = model.fast_generate(
    **messages.to('cuda'),
    max_new_tokens = sampling_params.max_tokens,
    temperature = sampling_params.temperature,
    top_p = sampling_params.top_p,
)
print(output)

tensor([[151644,   8948,    198,   2507,  11931,     13,   5443,    366,  19895,
            287,     29,    323,    366,   9217,     29,   9492,     13, 151645,
            198, 151644,    872,    198,  10879,     25,  12152,  14419,     33,
          72895, 151645,    198, 151644,  77091,    198,     27,  19895,    287,
          16357,   3409,    364,   6666,  14419,     33,  72895,      6,  17167,
            315,    220,     24,   3842,  11931,   3918,  19895,    287,    397,
             27,   9217,     29,     24,    522,   9217,     29, 151645]],
       device='cuda:0')


In [ ]:
decoded_output = tokenizer.decode(output[0])
print(decoded_output)

<|im_start|>system
Count letters. Use <reasoning> and <answer> tags.<|im_end|>
<|im_start|>user
Word: STRAWBERRY<|im_end|>
<|im_start|>assistant
<reasoning>The word 'STRAWBERRY' consists of 9 individual letters.</reasoning>
<answer>9</answer><|im_end|>


In [ ]:
test_prompt = "What is the capital of Japan?"
# Run the same generation code as above to verify 'Tokyo'

In [ ]:
from vllm import SamplingParams

# Test a word the model hasn't seen in the small training set
test_word = "STRAWBERRY"
prompt = [{"role": "system", "content": SYSTEM_PROMPT},
          {"role": "user", "content": f"Word: {test_word}"}]

messages = tokenizer.apply_chat_template(prompt, tokenize=True, add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True)
sampling_params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=1024)

# Generate the response
output = model.fast_generate(
    **messages.to('cuda'),
    max_new_tokens=sampling_params.max_tokens,
    temperature=sampling_params.temperature,
    top_p=sampling_params.top_p,
)
print(f"--- Model Output for {test_word} ---")
print(tokenizer.decode(output[0]))

--- Model Output for STRAWBERRY ---
<|im_start|>system
Count letters. Use <reasoning> and <answer> tags.<|im_end|>
<|im_start|>user
Word: STRAWBERRY<|im_end|>
<|im_start|>assistant
<reasoning>The task is to count the occurrence of each letter in the word 'STRAWBERRY'. I will analyze each letter one by one:
- S: 1
- T: 1
- R: 2
- A: 1
- W: 1
- B: 1
- Y: 1
<answer>The letter S occurs 1 time.
The letter T occurs 1 time.
The letter R occurs 2 times.
The letter A occurs 1 time.
The letter W occurs 1 time.
The letter B occurs 1 time.
The letter Y occurs 1 time.</answer><|im_end|>


In [ ]:
# Save the fine-tuned weights and tokenizer
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/chat_template.jinja',
 'lora_model/vocab.json',
 'lora_model/merges.txt',
 'lora_model/added_tokens.json',
 'lora_model/tokenizer.json')

In [ ]:
import shutil
import os

shutil.make_archive("submission", 'zip', base_dir=".")

print("Archive 'submission.zip' created.")

Archive 'submission.zip' created.


In [ ]:
import os
from google.colab import files
if os.path.exists("submission.zip"):
    files.download("submission.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>